In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [68]:
#load the dataset
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [69]:
## preprocess the data
### removing unnecessary columns
data=data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [70]:
label_encoder_gender=LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data["Gender"])


In [71]:
## Onehot encode for Geography
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[["Geography"]])

In [72]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out())

# combine coloumns
data=pd.concat([data.drop("Geography",axis=1),geo_encoded_df],axis=1)
data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [73]:
# save the label encoders and onehot encoders
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)    

In [74]:
# divide dataset into independent and dependent features
X=data.drop("Exited",axis=1)
y=data["Exited"]
# split the dataset into training and testing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
# scale the features
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [75]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

ANN Implementaion


In [76]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [77]:
# build our ANN model
model=Sequential([
    Dense(64,activation="relu",input_shape=(X_train_scaled.shape[1],)),
    Dense(32,activation="relu"),
    Dense(1,activation="sigmoid")
])

In [78]:
# compile the model
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])

In [79]:
# set up Tensorboard
log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)

In [80]:
# set up early stopping
early_stopping_callback=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [81]:
# training the model
history=model.fit(X_train_scaled,y_train,validation_data=(X_test_scaled,y_test),epochs=100,
                  callbacks=[early_stopping_callback,tensorflow_callback])

Epoch 1/100
250/250 [==============================] - 2s 5ms/step - loss: 0.4600 - accuracy: 0.7897 - val_loss: 0.3945 - val_accuracy: 0.8310
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3882 - accuracy: 0.8397 - val_loss: 0.3610 - val_accuracy: 0.8565
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3574 - accuracy: 0.8539 - val_loss: 0.3430 - val_accuracy: 0.8605
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3446 - accuracy: 0.8604 - val_loss: 0.3430 - val_accuracy: 0.8630
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3388 - accuracy: 0.8611 - val_loss: 0.3437 - val_accuracy: 0.8605
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3350 - accuracy: 0.8614 - val_loss: 0.3408 - val_accuracy: 0.8645
Epoch 7/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3322 - accuracy: 0.8650 - val_loss: 0.3437 - val_accuracy: 0.8600

In [82]:
model.save("model.h5")

/home/eyad/projects/ANN-classification/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [83]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [84]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 197305), started 0:09:37 ago. (Use '!kill 197305' to kill it.)